# CSMS AI Marine Scoring Engine — Full Pipeline with Embedding-Enhanced Aggregation

End-to-end analysis and testing notebook covering all 6 stages of the CSMS AI scoring engine:

1. **Extraction** — `ExtractorRouter` dispatches files to specialized extractors (PDF/OCR/Excel/Word/Image/CSV/Vision LLM)
2. **Aggregation** — `DocumentClassifier` + `PriorityAssigner` + `EvidenceAggregator` with **NEW: chunk + embed + SQLite semantic retrieval**
3. **Validation** — `ValidatorEngine` runs configured validation methods from `VR-{id}.json`
4. **Juror Scoring** — 3 independent AI Jurors (Compliance, Evidence, HSE) score on CSMS scale {0, 3, 6, 10}
5. **Consensus** — `ConsensusAgent` synthesizes equal-weighted verdict with LLM reasoning
6. **Recommend Score** — Final discrete CSMS score vs expected benchmark

### Enhancement: Semantic Aggregation
The original `EvidenceAggregator` concatenates all evidence into a flat string (truncated at 80K) and asks the LLM to summarize. This loses semantic relevance and wastes tokens.

**New approach:**
- **Chunk** extracted evidence into semantic units (~512 chars with 50 char overlap)
- **Embed** chunks using `text-embedding-3-small` (1536-dim, OpenAI)
- **Store** chunks + embeddings in SQLite
- **Semantic retrieval**: Query SQLite store using category scoring rules as query → retrieve top-K most relevant evidence chunks
- **Feed** only relevant evidence to jurors → better accuracy, lower token cost

---
## 1. Setup & Imports

In [ ]:
import os
import sys
import json
import re
import time
import hashlib
import struct
import mimetypes
import sqlite3
import logging
from typing import Dict, Any, List, Optional, Tuple
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import display, Markdown, HTML

sys.path.append(os.path.abspath('..'))
load_dotenv('../.env')

if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8', errors='replace')

PROJECT_ROOT = os.path.abspath('..')
MARINE_DIR = os.path.join(PROJECT_ROOT, 'test_submissions', 'marine')
KB_DIR = os.path.join(PROJECT_ROOT, 'knowledge_base', 'marine')
OUTPUT_DIR = os.path.join(PROJECT_ROOT, 'notebook_v2', 'outputs')
DB_PATH = os.path.join(OUTPUT_DIR, 'evidence_store.db')

os.makedirs(OUTPUT_DIR, exist_ok=True)

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
openai_client = OpenAI(api_key=OPENAI_API_KEY)

EMBEDDING_MODEL = 'text-embedding-3-small'
EMBEDDING_DIM = 1536
CHUNK_SIZE = 512
CHUNK_OVERLAP = 50
TOP_K = 15

print(f'Project root: {PROJECT_ROOT}')
print(f'Marine dir:   {MARINE_DIR}')
print(f'KB dir:       {KB_DIR}')
print(f'DB path:      {DB_PATH}')
print(f'Embedding:    {EMBEDDING_MODEL} ({EMBEDDING_DIM}-dim)')
print(f'API key:      {"loaded" if OPENAI_API_KEY else "MISSING"}')
print('Setup complete.')

---
## 2. Configuration

In [ ]:
TARGET_CATEGORY = '3.5'

JUROR_ROLE = 'reasoning_model'
CONSENSUS_ROLE = 'consensus_model'

USE_EMBEDDING_AGGREGATION = True
USE_ORIGINAL_PIPELINE = True
RUN_COMPARISON = True

categories = []
pattern = re.compile(r'^([0-9]+(\.[0-9]+)?)')
for item in sorted(os.listdir(MARINE_DIR)):
    full = os.path.join(MARINE_DIR, item)
    if os.path.isdir(full):
        m = pattern.match(item)
        if m:
            categories.append({'id': m.group(1), 'name': item, 'path': full})

target_folder = None
for c in categories:
    if c['id'] == TARGET_CATEGORY:
        target_folder = c
        break

print(f'Target category: {TARGET_CATEGORY}')
if target_folder:
    print(f'Folder: {target_folder["name"]}')
    print(f'Path:   {target_folder["path"]}')
else:
    print('WARNING: Target category folder not found!')
print(f'\nAll discovered categories ({len(categories)}):')
for c in categories:
    marker = ' <-- TARGET' if c['id'] == TARGET_CATEGORY else ''
    print(f'  {c["id"]:>5s}  {c["name"]}{marker}')

---
## 3. Architecture Analysis — Full Pipeline Deep Dive

### 3.1 Extraction Stage
`engine/extractors/router.py` — `ExtractorRouter` dispatches by file extension:

| Extension | Extractor | Strategy |
|-----------|-----------|----------|
| `.pdf` | `PDFExtractor` (PyMuPDF) → `OCRExtractor` fallback if <50 chars | Native text first, OCR if sparse |
| `.xlsx/.xls` | `ExcelExtractor` (pandas) | Sheet-by-sheet CSV dump |
| `.docx/.doc` | `WordExtractor` (python-docx) | Paragraph text |
| `.jpg/.png/.tif` | `ImageExtractor` | Vision LLM (Qwen3-VL) base64 |
| `.csv` | `CSVExtractor` | Raw CSV read |
| `.txt/.xml/.json` | `TextExtractor` | Raw text |
| `.msg` | `MSGExtractor` | Email body + attachments |
| `.ppt/.pptx` | `PPTExtractor` | Slide text |
| `.zip/.rar/.7z` | `ArchiveExtractor` | Recursive inner extraction |

After base extraction, `VisionExtractor` (Qwen3-VL via VLLM) enhances with structured JSON: `{document_type, summary, entities, tables, evidence, confidence}`.

Output per doc: `{status, evidence: [str], metadata, confidence, extractor_used, vision_model_used, characters_extracted, structured}`

### 3.2 Aggregation Stage
Three sub-components:
- **`DocumentClassifier`** — Rule-based filename keyword matching → Tier 1 (Deep Analysis) / Tier 2 (Structured) / Tier 3 (Mass Evidence) / Unclassified
- **`PriorityAssigner`** — Tier → priority integer 1-4
- **`EvidenceAggregator`** — LLM summarizes all extracted evidence into `{key_findings, document_types_found, missing_critical_info}`. Truncates at 4K per doc, 80K total.

**Weakness identified:** Flat string concatenation with hard truncation. No semantic relevance filtering. Token waste on irrelevant content.

### 3.3 Validation Stage
`engine/validators/engine.py` — `ValidatorEngine` runs configured methods from `VR-{id}.json`:

| Validator | Purpose |
|-----------|----------|
| `date_validation` | Check document dates are valid |
| `expiry_validator` | Check certificates not expired |
| `signature_check` | Verify signatures present |
| `keyword_search` | Match category-specific keywords |
| `completeness_validator` | Check all required fields |
| `competency_validator` | Verify competency records |
| `consistency_validator` | Cross-document consistency |
| `certificate_validator` | Validate certificate formats |

### 3.4 Juror Scoring Stage
Three independent AI Jurors, each receiving: `category_definition (KB)` + `aggregated_evidence` + `validation_results`

| Juror | Focus | Rubric |
|-------|-------|--------|
| `ComplianceJuror` | Certificates, procedures, regulatory compliance | Full/Substantial/Partial/Non-compliant |
| `EvidenceJuror` | Document quality, verification, completeness | Full/Solid/Minimal/No evidence |
| `HSEJuror` | HSE management, field implementation, certifications | Full/Systematic/Basic/No implementation |

Each outputs: `{score, confidence, reasoning}`. Score snapped to CSMS scale:
```python
def snap_to_csms_scale(val):
    if val < 1.5: return 0
    if val < 4.5: return 3
    if val < 8.0: return 6
    return 10
```

### 3.5 Consensus Stage
`engine/consensus/consensus_agent.py` — `ConsensusAgent`:
1. Equal-weight average of 3 juror scores
2. Snap to CSMS discrete scale {0, 3, 6, 10}
3. LLM synthesizes reasoning from all 3 juror verdicts
4. `requires_review = True` if max juror score difference > 4

Output: `{recommended_score, confidence, requires_review, reasoning}`

### 3.6 Recommend Score
Final `snap_to_csms_scale()` on consensus output. Compared against `expected_score` from `expected_result.json`.

---
## 4. Knowledge Base Analysis

Each category has 3 KB files:
- `categories/{id}.json` — Category definition, required evidence list
- `scoring_rules/SR-{id}.json` — Detailed scoring rubric per score level (0, 3, 6, 10)
- `validation_rules/VR-{id}.json` — Validation methods + mandatory fields + AI checkpoint keywords

In [ ]:
def load_kb(category_id: str) -> Dict[str, Any]:
    clean_id = re.match(r'^[0-9]+(\.[0-9]+)?', str(category_id).strip())
    clean_id = clean_id.group(0) if clean_id else str(category_id).strip()
    result = {}
    for kind, sub, prefix in [('category', 'categories', ''), ('scoring_rules', 'scoring_rules', 'SR-'), ('validation_rules', 'validation_rules', 'VR-')]:
        fpath = os.path.join(KB_DIR, sub, f'{prefix}{clean_id}.json')
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                result[kind] = json.load(f)
        else:
            result[kind] = {}
    return result

kb = load_kb(TARGET_CATEGORY)

print(f'=== Category Definition ({TARGET_CATEGORY}) ===')
print(json.dumps(kb['category'], indent=2, ensure_ascii=False)[:2000])

print(f'\n=== Scoring Rules ({TARGET_CATEGORY}) ===')
sr = kb['scoring_rules']
for level in ['0', '3', '6', '10']:
    desc = sr.get('score_levels', {}).get(level, 'N/A')
    print(f'\n  Score {level}:')
    print(f'    {desc[:300]}')

print(f'\n  Reasoning Hint: {sr.get("reasoning_hint", "N/A")[:500]}')

print(f'\n=== Validation Rules ({TARGET_CATEGORY}) ===')
vr = kb['validation_rules']
print(f'  Methods: {vr.get("validation_methods", [])}')
print(f'  Keywords: {vr.get("ai_checkpoint_keywords", [])}')
print(f'  Mandatory Fields: {vr.get("mandatory_fields", [])}')

---
## 5. Stage 1/6 — Extraction

Using the existing `ExtractorRouter` to extract all documents from the target category folder.

In [ ]:
from engine.extractors.router import ExtractorRouter
from engine.aggregators.document_classifier import DocumentClassifier
from engine.aggregators.priority_assigner import PriorityAssigner

extractor = ExtractorRouter()
classifier = DocumentClassifier()
assigner = PriorityAssigner()

IGNORE_FILES = {'expected_result.json', 'readme.md', '.keep', '.placeholder', 'ignore_me.txt', 'thumbs.db', '.ds_store'}

extracted_docs = []
extraction_log = []

if target_folder:
    folder_path = target_folder['path']
    for root, dirs, files in os.walk(folder_path):
        for file in sorted(files):
            if file.lower() in IGNORE_FILES:
                continue
            fpath = os.path.join(root, file)
            ext = os.path.splitext(file)[1].lower()
            size = os.path.getsize(fpath) if os.path.exists(fpath) else 0
            
            t0 = time.time()
            try:
                doc_data = extractor.extract_document(fpath)
                elapsed = round(time.time() - t0, 2)
                
                cls = classifier.classify(fpath, doc_data.get('metadata', {}))
                pri = assigner.assign_priority(cls)
                doc_data['file'] = file
                doc_data['file_path'] = fpath
                doc_data['classification'] = cls
                doc_data['priority'] = pri
                doc_data['extraction_time'] = elapsed
                
                extracted_docs.append(doc_data)
                
                chars = doc_data.get('characters_extracted', 0)
                status = doc_data.get('status', 'ok')
                ext_used = doc_data.get('extractor_used', '?')
                vis = doc_data.get('vision_model_used', 'none')
                
                extraction_log.append({
                    'file': file, 'ext': ext, 'size': size, 'status': status,
                    'extractor': ext_used, 'vision': vis, 'chars': chars,
                    'tier': cls, 'priority': pri, 'time': elapsed
                })
                print(f'  [{status.upper():>4s}] {file:<55s} | {chars:>6d} chars | {cls:<30s} | {elapsed:.1f}s')
            except Exception as e:
                print(f'  [FAIL] {file}: {e}')
                extraction_log.append({'file': file, 'ext': ext, 'size': size, 'status': 'error', 'error': str(e)})

df_ext = pd.DataFrame(extraction_log)
print(f'\nTotal documents extracted: {len(extracted_docs)}')
total_chars = sum(d.get('characters_extracted', 0) for d in extracted_docs)
total_evidence = sum(len(d.get('evidence', [])) for d in extracted_docs)
print(f'Total characters: {total_chars:,}')
print(f'Total evidence chunks: {total_evidence}')
display(df_ext)

---
## 6. Stage 2a — Chunking Strategy

**Why chunk?** The current `EvidenceAggregator` dumps all evidence into one massive string (up to 80K chars) for the LLM to summarize. This:
1. Wastes tokens on irrelevant content
2. Loses semantic signal when hard-truncated
3. Makes it impossible to query specific evidence later

**Chunking approach:**
- Break each document's evidence into overlapping windows of `CHUNK_SIZE` chars
- Each chunk retains metadata: source file, category, document type, tier classification
- Chunks become the atomic unit for embedding and retrieval

In [ ]:
def chunk_text(text: str, chunk_size: int = CHUNK_SIZE, overlap: int = CHUNK_OVERLAP) -> List[str]:
    if not text or not text.strip():
        return []
    text = text.strip()
    if len(text) <= chunk_size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk.strip():
            chunks.append(chunk.strip())
        if end >= len(text):
            break
        start = end - overlap
    return chunks

def create_chunks_from_docs(extracted_docs: List[Dict[str, Any]], category_id: str) -> List[Dict[str, Any]]:
    all_chunks = []
    for doc in extracted_docs:
        filename = doc.get('file', 'unknown')
        doc_type = doc.get('document_type', 'unknown')
        tier = doc.get('classification', 'Unclassified')
        priority = doc.get('priority', 4)
        
        evidence_list = doc.get('evidence', [])
        for ev_idx, evidence in enumerate(evidence_list):
            ev_str = str(evidence)
            if not ev_str.strip():
                continue
            text_chunks = chunk_text(ev_str)
            for ch_idx, chunk_text_val in enumerate(text_chunks):
                chunk_id = hashlib.sha256(f'{category_id}|{filename}|{ev_idx}|{ch_idx}'.encode()).hexdigest()[:16]
                all_chunks.append({
                    'chunk_id': chunk_id,
                    'category_id': category_id,
                    'filename': filename,
                    'doc_type': doc_type,
                    'tier': tier,
                    'priority': priority,
                    'evidence_idx': ev_idx,
                    'chunk_idx': ch_idx,
                    'text': chunk_text_val,
                    'char_count': len(chunk_text_val)
                })
    return all_chunks

chunks = create_chunks_from_docs(extracted_docs, TARGET_CATEGORY)

print(f'Chunking results for Category {TARGET_CATEGORY}:')
print(f'  Documents:         {len(extracted_docs)}')
print(f'  Evidence items:    {total_evidence}')
print(f'  Chunks generated:  {len(chunks)}')
print(f'  Chunk size:        {CHUNK_SIZE} chars (overlap: {CHUNK_OVERLAP})')
print(f'  Total chunk chars: {sum(c["char_count"] for c in chunks):,}')

if chunks:
    print(f'\n--- Sample chunks (first 3) ---')
    for i, ch in enumerate(chunks[:3]):
        print(f'\n  Chunk #{i+1} [{ch["chunk_id"]}] from {ch["filename"]}')
        print(f'    Type: {ch["doc_type"]} | Tier: {ch["tier"]} | Priority: {ch["priority"]}')
        print(f'    Text ({ch["char_count"]} chars): {ch["text"][:200]}...')

df_chunks = pd.DataFrame([{k: v for k, v in c.items() if k != 'text'} for c in chunks])
if len(df_chunks) > 0:
    print(f'\n--- Chunk distribution by file ---')
    display(df_chunks.groupby('filename').agg(chunk_count=('chunk_id', 'count'), total_chars=('char_count', 'sum')).reset_index())

---
## 7. Stage 2b — Embedding with OpenAI text-embedding-3-small

Each chunk is embedded into a 1536-dimensional vector using `text-embedding-3-small`.

Why this model:
- Cost: $0.02 / 1M tokens (very cheap)
- Performance: Strong on MTEB benchmarks for retrieval
- Size: 1536 dims — fits comfortably in SQLite
- Multilingual: Handles Indonesian/English mixed content well

In [ ]:
def embed_texts(texts: List[str], model: str = EMBEDDING_MODEL, batch_size: int = 100) -> List[List[float]]:
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        cleaned = [t.replace('\n', ' ').strip() or 'empty' for t in batch]
        try:
            response = openai_client.embeddings.create(model=model, input=cleaned)
            batch_embeddings = [item.embedding for item in response.data]
            all_embeddings.extend(batch_embeddings)
            print(f'  Batch {i // batch_size + 1}: embedded {len(batch)} chunks')
        except Exception as e:
            print(f'  ERROR embedding batch {i // batch_size + 1}: {e}')
            all_embeddings.extend([[0.0] * EMBEDDING_DIM] * len(batch))
    return all_embeddings

if chunks:
    t0 = time.time()
    chunk_texts = [c['text'] for c in chunks]
    embeddings = embed_texts(chunk_texts)
    embed_time = round(time.time() - t0, 2)
    
    for i, emb in enumerate(embeddings):
        chunks[i]['embedding'] = emb
    
    print(f'\nEmbedding complete: {len(embeddings)} chunks in {embed_time}s')
    if embeddings and any(embeddings[0]):
        sample = embeddings[0]
        print(f'Sample embedding (first 10 dims): {sample[:10]}')
        print(f'Embedding norm: {np.linalg.norm(sample):.4f}')
else:
    embeddings = []
    print('No chunks to embed.')

---
## 8. Stage 2c — SQLite Vector Store

Store chunks + embeddings in SQLite for:
1. **Persistent storage** — no external vector DB dependency
2. **Fast cosine similarity** — using numpy for vector math
3. **Metadata filtering** — SQL WHERE clauses for category/file/tier filtering
4. **FTS5 full-text search** — bonus keyword search alongside semantic search

In [ ]:
def init_evidence_db(db_path: str) -> sqlite3.Connection:
    if os.path.exists(db_path):
        os.remove(db_path)
    conn = sqlite3.connect(db_path)
    conn.execute('PRAGMA journal_mode=WAL')
    conn.execute('PRAGMA synchronous=NORMAL')
    
    conn.execute('''
        CREATE TABLE IF NOT EXISTS chunks (
            chunk_id TEXT PRIMARY KEY,
            category_id TEXT NOT NULL,
            filename TEXT NOT NULL,
            doc_type TEXT,
            tier TEXT,
            priority INTEGER,
            evidence_idx INTEGER,
            chunk_idx INTEGER,
            text TEXT NOT NULL,
            char_count INTEGER,
            embedding BLOB,
            created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    
    conn.execute('CREATE INDEX IF NOT EXISTS idx_category ON chunks(category_id)')
    conn.execute('CREATE INDEX IF NOT EXISTS idx_filename ON chunks(filename)')
    conn.execute('CREATE INDEX IF NOT EXISTS idx_tier ON chunks(tier)')
    conn.execute('CREATE INDEX IF NOT EXISTS idx_priority ON chunks(priority)')
    
    conn.execute('''
        CREATE VIRTUAL TABLE IF NOT EXISTS chunks_fts USING fts5(
            chunk_id, text, filename, category_id,
            content='chunks',
            content_rowid='rowid'
        )
    ''')
    
    conn.execute('''
        CREATE TRIGGER IF NOT EXISTS chunks_ai AFTER INSERT ON chunks BEGIN
            INSERT INTO chunks_fts(chunk_id, text, filename, category_id)
            VALUES (NEW.chunk_id, NEW.text, NEW.filename, NEW.category_id);
        END
    ''')
    
    conn.commit()
    return conn

def embedding_to_blob(embedding: List[float]) -> bytes:
    return struct.pack(f'{len(embedding)}f', *embedding)

def blob_to_embedding(blob: bytes) -> List[float]:
    n = len(blob) // 4
    return list(struct.unpack(f'{n}f', blob))

def store_chunks_in_db(conn: sqlite3.Connection, chunks: List[Dict[str, Any]]) -> int:
    cursor = conn.cursor()
    inserted = 0
    for ch in chunks:
        emb_blob = embedding_to_blob(ch['embedding']) if ch.get('embedding') else None
        try:
            cursor.execute('''
                INSERT OR REPLACE INTO chunks 
                (chunk_id, category_id, filename, doc_type, tier, priority, evidence_idx, chunk_idx, text, char_count, embedding)
                VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            ''', (
                ch['chunk_id'], ch['category_id'], ch['filename'], ch['doc_type'],
                ch['tier'], ch['priority'], ch['evidence_idx'], ch['chunk_idx'],
                ch['text'], ch['char_count'], emb_blob
            ))
            inserted += 1
        except Exception as e:
            print(f'  DB insert error for {ch["chunk_id"]}: {e}')
    conn.commit()
    return inserted

conn = init_evidence_db(DB_PATH)
n_stored = store_chunks_in_db(conn, chunks)
print(f'Stored {n_stored} chunks in SQLite: {DB_PATH}')

row = conn.execute('SELECT COUNT(*), SUM(char_count) FROM chunks').fetchone()
print(f'DB stats: {row[0]} chunks, {row[1]:,} total characters')

sample_row = conn.execute('SELECT chunk_id, category_id, filename, tier, priority, char_count FROM chunks LIMIT 3').fetchall()
for r in sample_row:
    print(f'  {r}')

---
## 9. Stage 2d — Semantic Retrieval

Use category scoring rules as query → find top-K most relevant evidence chunks.

**Query construction:** Combine scoring rubric + reasoning hints + mandatory fields from KB into a semantic query.

**Hybrid search:**
1. Semantic: Cosine similarity between query embedding and chunk embeddings
2. Keyword (FTS5): Full-text match on AI checkpoint keywords
3. Reciprocal Rank Fusion (RRF): Combine both rankings

In [ ]:
def build_query_from_kb(kb: Dict[str, Any]) -> str:
    parts = []
    cat = kb.get('category', {})
    sr = kb.get('scoring_rules', {})
    vr = kb.get('validation_rules', {})
    
    if cat.get('category_name'):
        parts.append(f'Category: {cat["category_name"]}')
    if cat.get('required_evidence'):
        parts.append(f'Required evidence: {", ".join(cat["required_evidence"])}')
    if cat.get('description'):
        parts.append(f'Description: {cat["description"]}')
    
    if sr.get('reasoning_hint'):
        parts.append(f'Scoring criteria: {sr["reasoning_hint"]}')
    
    if vr.get('ai_checkpoint_keywords'):
        parts.append(f'Keywords: {", ".join(vr["ai_checkpoint_keywords"])}')
    if vr.get('mandatory_fields'):
        parts.append(f'Mandatory: {", ".join(vr["mandatory_fields"])}')
    
    return '\n'.join(parts)

def semantic_search(conn: sqlite3.Connection, query_embedding: List[float], category_id: str = None, top_k: int = TOP_K) -> List[Dict[str, Any]]:
    query = 'SELECT chunk_id, category_id, filename, doc_type, tier, priority, text, char_count, embedding FROM chunks'
    params = []
    if category_id:
        query += ' WHERE category_id = ?'
        params.append(category_id)
    
    rows = conn.execute(query, params).fetchall()
    if not rows:
        return []
    
    query_vec = np.array(query_embedding, dtype=np.float32)
    query_norm = np.linalg.norm(query_vec)
    if query_norm == 0:
        return []
    
    scored = []
    for row in rows:
        chunk_id, cat_id, filename, doc_type, tier, priority, text, char_count, emb_blob = row
        if emb_blob is None:
            continue
        chunk_vec = np.array(blob_to_embedding(emb_blob), dtype=np.float32)
        chunk_norm = np.linalg.norm(chunk_vec)
        if chunk_norm == 0:
            continue
        cosine_sim = float(np.dot(query_vec, chunk_vec) / (query_norm * chunk_norm))
        scored.append({
            'chunk_id': chunk_id,
            'category_id': cat_id,
            'filename': filename,
            'doc_type': doc_type,
            'tier': tier,
            'priority': priority,
            'text': text,
            'char_count': char_count,
            'semantic_score': round(cosine_sim, 6)
        })
    
    scored.sort(key=lambda x: x['semantic_score'], reverse=True)
    return scored[:top_k]

def keyword_search(conn: sqlite3.Connection, keywords: List[str], category_id: str = None, top_k: int = TOP_K) -> List[Dict[str, Any]]:
    if not keywords:
        return []
    query_str = ' OR '.join(keywords)
    sql = '''
        SELECT c.chunk_id, c.category_id, c.filename, c.doc_type, c.tier, c.priority, c.text, c.char_count
        FROM chunks_fts fts
        JOIN chunks c ON c.chunk_id = fts.chunk_id
        WHERE chunks_fts MATCH ?
    '''
    params = [query_str]
    if category_id:
        sql += ' AND fts.category_id = ?'
        params.append(category_id)
    sql += f' LIMIT {top_k * 2}'
    
    try:
        rows = conn.execute(sql, params).fetchall()
    except Exception as e:
        print(f'  FTS5 search error: {e}')
        return []
    
    results = []
    for i, row in enumerate(rows):
        results.append({
            'chunk_id': row[0], 'category_id': row[1], 'filename': row[2],
            'doc_type': row[3], 'tier': row[4], 'priority': row[5],
            'text': row[6], 'char_count': row[7],
            'keyword_rank': i + 1
        })
    return results

def hybrid_search(conn: sqlite3.Connection, query_text: str, keywords: List[str], category_id: str = None, top_k: int = TOP_K, k_rrf: int = 60) -> List[Dict[str, Any]]:
    query_emb = embed_texts([query_text])[0]
    
    semantic_results = semantic_search(conn, query_emb, category_id, top_k=top_k * 2)
    keyword_results = keyword_search(conn, keywords, category_id, top_k=top_k * 2)
    
    scores = {}
    for rank, item in enumerate(semantic_results):
        cid = item['chunk_id']
        scores[cid] = scores.get(cid, {'data': item, 'rrf': 0.0})
        scores[cid]['rrf'] += 1.0 / (k_rrf + rank + 1)
    
    for rank, item in enumerate(keyword_results):
        cid = item['chunk_id']
        if cid not in scores:
            scores[cid] = {'data': item, 'rrf': 0.0}
        scores[cid]['rrf'] += 1.0 / (k_rrf + rank + 1)
    
    ranked = sorted(scores.values(), key=lambda x: x['rrf'], reverse=True)
    
    results = []
    for item in ranked[:top_k]:
        entry = item['data'].copy()
        entry['rrf_score'] = round(item['rrf'], 6)
        results.append(entry)
    
    return results

query_text = build_query_from_kb(kb)
keywords = kb.get('validation_rules', {}).get('ai_checkpoint_keywords', [])

print(f'=== Semantic Query for Category {TARGET_CATEGORY} ===')
print(f'Query text:\n{query_text}\n')
print(f'Keywords: {keywords}')

relevant_chunks = hybrid_search(conn, query_text, keywords, category_id=TARGET_CATEGORY, top_k=TOP_K)

print(f'\n=== Top {len(relevant_chunks)} Relevant Chunks ===')
for i, ch in enumerate(relevant_chunks):
    sem = ch.get('semantic_score', 'N/A')
    rrf = ch.get('rrf_score', 'N/A')
    print(f'\n  [{i+1}] {ch["filename"]} (Tier: {ch["tier"]}, Sem: {sem}, RRF: {rrf})')
    print(f'      {ch["text"][:200]}...')

---
## 10. Stage 2e — Enhanced Aggregation via Semantic Retrieval

Compare original flat aggregation vs embedding-enhanced aggregation.

In [ ]:
from engine.aggregators.evidence_aggregator import EvidenceAggregator

print('=== Original Aggregation (flat concat + LLM summarize) ===')
t0 = time.time()
original_aggregator = EvidenceAggregator()
original_aggregated = original_aggregator.aggregate(extracted_docs)
original_agg_time = round(time.time() - t0, 2)

print(f'Time: {original_agg_time}s')
print(f'Key findings ({len(original_aggregated.get("key_findings", []))}):')
for f in original_aggregated.get('key_findings', [])[:5]:
    print(f'  - {f[:150]}')
print(f'Doc types: {original_aggregated.get("document_types_found", [])}')
print(f'Missing critical info: {original_aggregated.get("missing_critical_info")}')
print(f'\nFull aggregated JSON ({len(json.dumps(original_aggregated))} chars):')
print(json.dumps(original_aggregated, indent=2, ensure_ascii=False)[:1500])

In [ ]:
def build_enhanced_aggregation(relevant_chunks: List[Dict[str, Any]], kb: Dict[str, Any]) -> Dict[str, Any]:
    if not relevant_chunks:
        return {'key_findings': [], 'document_types_found': [], 'missing_critical_info': True, 'source_chunks': 0}
    
    chunk_texts = [f'[{ch["filename"]}] {ch["text"]}' for ch in relevant_chunks]
    combined = '\n\n'.join(chunk_texts)
    
    if len(combined) > 12000:
        combined = combined[:12000] + '\n...[truncated]'
    
    schema = {
        'type': 'object',
        'properties': {
            'key_findings': {'type': 'array', 'items': {'type': 'string'}, 'description': 'Most important facts from retrieved evidence chunks'},
            'document_types_found': {'type': 'array', 'items': {'type': 'string'}},
            'missing_critical_info': {'type': 'boolean', 'description': 'True if standard critical HSE information seems missing'},
            'evidence_quality_score': {'type': 'number', 'description': '0.0-1.0 quality score of available evidence'},
            'relevant_evidence_count': {'type': 'integer'}
        },
        'required': ['key_findings', 'document_types_found', 'missing_critical_info']
    }
    
    scoring_hint = kb.get('scoring_rules', {}).get('reasoning_hint', '')
    mandatory = kb.get('validation_rules', {}).get('mandatory_fields', [])
    
    prompt = f'''Aggregate the following semantically-retrieved evidence chunks for CSMS category evaluation.

Scoring criteria hint: {scoring_hint}
Mandatory fields to check: {mandatory}

Retrieved Evidence ({len(relevant_chunks)} chunks):
{combined}

Identify key findings, document types found, and whether critical information is missing.'''
    
    system_prompt = 'You are the Evidence Aggregator Agent. Summarize retrieved evidence precisely. Focus on compliance-relevant facts.'
    
    try:
        messages = [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': prompt + f'\n\nRespond with JSON: {json.dumps(schema)}'}
        ]
        response = openai_client.chat.completions.create(
            model='gpt-4o-mini',
            messages=messages,
            temperature=0.0,
            response_format={'type': 'json_object'}
        )
        content = response.choices[0].message.content.strip()
        if content.startswith('```json'):
            content = content[7:]
        if content.startswith('```'):
            content = content[3:]
        if content.endswith('```'):
            content = content[:-3]
        result = json.loads(content.strip())
    except Exception as e:
        print(f'  LLM aggregation error: {e}')
        result = {
            'key_findings': [ch['text'][:200] for ch in relevant_chunks[:10]],
            'document_types_found': list(set(ch.get('doc_type', 'unknown') for ch in relevant_chunks)),
            'missing_critical_info': False
        }
    
    result['source_chunks'] = len(relevant_chunks)
    result['retrieval_method'] = 'hybrid_semantic_keyword'
    return result

print('=== Enhanced Aggregation (semantic retrieval + LLM summarize) ===')
t0 = time.time()
enhanced_aggregated = build_enhanced_aggregation(relevant_chunks, kb)
enhanced_agg_time = round(time.time() - t0, 2)

print(f'Time: {enhanced_agg_time}s')
print(f'Source chunks: {enhanced_aggregated.get("source_chunks", 0)}')
print(f'Retrieval: {enhanced_aggregated.get("retrieval_method", "N/A")}')
print(f'Key findings ({len(enhanced_aggregated.get("key_findings", []))}):')
for f in enhanced_aggregated.get('key_findings', [])[:5]:
    print(f'  - {f[:150]}')
print(f'Doc types: {enhanced_aggregated.get("document_types_found", [])}')
print(f'Missing critical info: {enhanced_aggregated.get("missing_critical_info")}')
print(f'\nFull enhanced JSON:')
print(json.dumps(enhanced_aggregated, indent=2, ensure_ascii=False)[:2000])

In [ ]:
print('=== Aggregation Comparison ===')
print(f'{"Metric":<35s} {"Original":>15s} {"Enhanced":>15s}')
print('-' * 65)

orig_findings = len(original_aggregated.get('key_findings', []))
enh_findings = len(enhanced_aggregated.get('key_findings', []))
orig_json_size = len(json.dumps(original_aggregated))
enh_json_size = len(json.dumps(enhanced_aggregated))

print(f'{"LLM aggregation time":<35s} {original_agg_time:>14.2f}s {enhanced_agg_time:>14.2f}s')
print(f'{"Key findings count":<35s} {orig_findings:>15d} {enh_findings:>15d}')
print(f'{"Aggregated JSON size":<35s} {orig_json_size:>15d} {enh_json_size:>15d}')
print(f'{"Document types found":<35s} {len(original_aggregated.get("document_types_found", [])):>15d} {len(enhanced_aggregated.get("document_types_found", [])):>15d}')
print(f'{"Missing critical info":<35s} {str(original_aggregated.get("missing_critical_info")):>15s} {str(enhanced_aggregated.get("missing_critical_info")):>15s}')
print(f'{"Input evidence chunks":<35s} {"all (flat)":>15s} {enhanced_aggregated.get("source_chunks", 0):>15d}')

print(f'\nEnhanced aggregation feeds only top-{TOP_K} semantically relevant chunks')
print(f'instead of all {total_evidence} evidence items, reducing noise and token cost.')

---
## 11. Stage 3/6 — Validation

Running configured validation methods from `VR-{category_id}.json`.

In [ ]:
from engine.validators.engine import ValidatorEngine

validator = ValidatorEngine()
validation_methods = kb.get('validation_rules', {}).get('validation_methods', [])
ev_list = [str(d.get('evidence', [])) for d in extracted_docs]
extracted_data = {'evidence': ev_list}

t0 = time.time()
validation_results = validator.run_validations(validation_methods, extracted_data, kb['validation_rules'])
val_time = round(time.time() - t0, 3)

print(f'=== Validation Results for Category {TARGET_CATEGORY} ===')
print(f'Methods: {validation_methods}')
print(f'Time: {val_time}s\n')

for method, result in validation_results.items():
    is_valid = result.get('is_valid', result.get('passed', False))
    conf = result.get('confidence', 0.0)
    reasoning = result.get('reasoning', result.get('details', 'N/A'))
    status = 'PASS' if is_valid else 'FAIL'
    print(f'  [{status}] {method} (conf: {conf:.2f})')
    if isinstance(reasoning, str):
        print(f'         {reasoning[:200]}')
    elif isinstance(reasoning, list):
        for r in reasoning[:3]:
            print(f'         {str(r)[:200]}')
    print()

---
## 12. Stage 4/6 — AI Juror Scoring

Three independent jurors evaluate using:
- Knowledge Base (category definition + scoring rules + validation rules)
- Aggregated evidence (original OR enhanced)
- Validation results

Each juror returns `{score, confidence, reasoning}`, snapped to CSMS scale {0, 3, 6, 10}.

In [ ]:
from engine.jurors.compliance_juror import ComplianceJuror
from engine.jurors.evidence_juror import EvidenceJuror
from engine.jurors.hse_juror import HSEJuror
from engine.consensus.consensus_agent import ConsensusAgent

def snap_to_csms_scale(val):
    try:
        v = float(val)
    except Exception:
        return 6
    if v < 1.5: return 0
    if v < 4.5: return 3
    if v < 8.0: return 6
    return 10

def run_jurors(kb, aggregated_evidence, validation_results, label=''):
    comp_juror = ComplianceJuror(role=JUROR_ROLE)
    ev_juror = EvidenceJuror(role=JUROR_ROLE)
    hse_juror = HSEJuror(role=JUROR_ROLE)
    
    results = {}
    
    t0 = time.time()
    comp_res = comp_juror.evaluate(kb, aggregated_evidence, validation_results)
    results['compliance'] = {'result': comp_res, 'score': snap_to_csms_scale(comp_res.get('score', 6)), 'time': round(time.time() - t0, 2)}
    
    t0 = time.time()
    ev_res = ev_juror.evaluate(kb, aggregated_evidence, validation_results)
    results['evidence'] = {'result': ev_res, 'score': snap_to_csms_scale(ev_res.get('score', 6)), 'time': round(time.time() - t0, 2)}
    
    t0 = time.time()
    hse_res = hse_juror.evaluate(kb, aggregated_evidence, validation_results)
    results['hse'] = {'result': hse_res, 'score': snap_to_csms_scale(hse_res.get('score', 6)), 'time': round(time.time() - t0, 2)}
    
    print(f'\n=== {label} Juror Results ===')
    for juror_name, data in results.items():
        r = data['result']
        score = data['score']
        conf = r.get('confidence', 0.0)
        reasoning = r.get('reasoning', [])
        if isinstance(reasoning, list):
            reasoning_str = '; '.join(reasoning[:3])
        else:
            reasoning_str = str(reasoning)[:300]
        print(f'  {juror_name.title():15s} | Score: {score:2d}/10 | Conf: {conf:.2f} | Time: {data["time"]}s')
        print(f'    Reasoning: {reasoning_str[:250]}')
    
    return results

print('Running jurors with ORIGINAL aggregation...')
original_juror_results = run_jurors(kb, original_aggregated, validation_results, label='Original')

if USE_EMBEDDING_AGGREGATION:
    print('\n\nRunning jurors with ENHANCED aggregation...')
    enhanced_juror_results = run_jurors(kb, enhanced_aggregated, validation_results, label='Enhanced')

---
## 13. Stage 5/6 — Consensus Synthesis

In [ ]:
def run_consensus(juror_results: Dict[str, Any], kb: Dict[str, Any], label: str = '') -> Dict[str, Any]:
    comp_res = juror_results['compliance']['result']
    ev_res = juror_results['evidence']['result']
    hse_res = juror_results['hse']['result']
    
    consensus = ConsensusAgent(role=CONSENSUS_ROLE)
    t0 = time.time()
    final = consensus.resolve(comp_res, ev_res, hse_res, kb)
    elapsed = round(time.time() - t0, 2)
    
    recommended = snap_to_csms_scale(final.get('recommended_score', 6))
    confidence = final.get('confidence', 0.0)
    requires_review = final.get('requires_review', False)
    reasoning = final.get('reasoning', 'N/A')
    
    scores = [juror_results['compliance']['score'], juror_results['evidence']['score'], juror_results['hse']['score']]
    
    print(f'\n=== {label} Consensus ===')
    print(f'  Juror scores:       Compliance={scores[0]}, Evidence={scores[1]}, HSE={scores[2]}')
    print(f'  Consensus score:    {recommended}/10')
    print(f'  Confidence:         {confidence:.2f}')
    print(f'  Requires review:    {requires_review}')
    print(f'  Consensus time:     {elapsed}s')
    print(f'  Reasoning:          {str(reasoning)[:400]}')
    
    return {
        'recommended_score': recommended,
        'confidence': confidence,
        'requires_review': requires_review,
        'reasoning': reasoning,
        'juror_scores': scores,
        'time': elapsed
    }

original_consensus = run_consensus(original_juror_results, kb, label='Original')

if USE_EMBEDDING_AGGREGATION:
    enhanced_consensus = run_consensus(enhanced_juror_results, kb, label='Enhanced')

---
## 14. Stage 6/6 — Recommend Score & Final Report

In [ ]:
expected_file = os.path.join(target_folder['path'], 'expected_result.json') if target_folder else None
expected_score = 10
if expected_file and os.path.exists(expected_file):
    with open(expected_file, 'r') as f:
        expected_score = json.load(f).get('expected_score', 10)

print('=' * 80)
print(f'CSMS AI SCORING — FINAL REPORT: Category {TARGET_CATEGORY}')
print(f'Folder: {target_folder["name"] if target_folder else "N/A"}')
print('=' * 80)

print(f'\n--- Input Summary ---')
print(f'  Documents:       {len(extracted_docs)}')
print(f'  Total chars:     {total_chars:,}')
print(f'  Evidence items:  {total_evidence}')
print(f'  Chunks created:  {len(chunks)}')
print(f'  Expected score:  {expected_score}/10')

print(f'\n--- Original Pipeline ---')
print(f'  Predicted:  {original_consensus["recommended_score"]}/10')
print(f'  Confidence: {original_consensus["confidence"]:.2f}')
match_orig = original_consensus['recommended_score'] == expected_score
print(f'  Correct:    {"YES" if match_orig else "NO"} (expected {expected_score})')

if USE_EMBEDDING_AGGREGATION:
    print(f'\n--- Enhanced Pipeline (Embedding + Semantic Retrieval) ---')
    print(f'  Predicted:  {enhanced_consensus["recommended_score"]}/10')
    print(f'  Confidence: {enhanced_consensus["confidence"]:.2f}')
    match_enh = enhanced_consensus['recommended_score'] == expected_score
    print(f'  Correct:    {"YES" if match_enh else "NO"} (expected {expected_score})')

print(f'\n--- Comparison ---')
print(f'{"Metric":<35s} {"Original":>15s} {"Enhanced":>15s}')
print('-' * 65)
print(f'{"Predicted score":<35s} {original_consensus["recommended_score"]:>15d} {enhanced_consensus.get("recommended_score", "N/A"):>15}')
print(f'{"Score correct":<35s} {str(match_orig):>15s} {str(match_enh) if USE_EMBEDDING_AGGREGATION else "N/A":>15s}')
print(f'{"Confidence":<35s} {original_consensus["confidence"]:>15.2f} {enhanced_consensus.get("confidence", 0):>15.2f}')
print(f'{"Juror compliance score":<35s} {original_juror_results["compliance"]["score"]:>15d} {enhanced_juror_results.get("compliance", {}).get("score", "N/A"):>15}')
print(f'{"Juror evidence score":<35s} {original_juror_results["evidence"]["score"]:>15d} {enhanced_juror_results.get("evidence", {}).get("score", "N/A"):>15}')
print(f'{"Juror HSE score":<35s} {original_juror_results["hse"]["score"]:>15d} {enhanced_juror_results.get("hse", {}).get("score", "N/A"):>15}')

print('\n' + '=' * 80)

---
## 15. Batch Test — All Categories

Run the enhanced pipeline across all discovered categories and compare with original.

In [ ]:
def score_category_enhanced(category_id: str, folder_path: str, conn: sqlite3.Connection) -> Dict[str, Any]:
    t_start = time.time()
    
    kb = load_kb(category_id)
    
    ext_router = ExtractorRouter()
    doc_classifier = DocumentClassifier()
    pri_assigner = PriorityAssigner()
    
    extracted = []
    for root, _, files in os.walk(folder_path):
        for file in sorted(files):
            if file.lower() in IGNORE_FILES:
                continue
            fpath = os.path.join(root, file)
            try:
                doc_data = ext_router.extract_document(fpath)
                if doc_data.get('status') != 'error':
                    doc_data['file'] = file
                    cls = doc_classifier.classify(fpath, doc_data.get('metadata', {}))
                    doc_data['classification'] = cls
                    doc_data['priority'] = pri_assigner.assign_priority(cls)
                    extracted.append(doc_data)
            except Exception as e:
                pass
    
    cat_chunks = create_chunks_from_docs(extracted, category_id)
    
    if cat_chunks:
        texts = [c['text'] for c in cat_chunks]
        embs = embed_texts(texts)
        for i, emb in enumerate(embs):
            cat_chunks[i]['embedding'] = emb
        store_chunks_in_db(conn, cat_chunks)
    
    query_text = build_query_from_kb(kb)
    keywords = kb.get('validation_rules', {}).get('ai_checkpoint_keywords', [])
    relevant = hybrid_search(conn, query_text, keywords, category_id=category_id, top_k=TOP_K) if cat_chunks else []
    
    enhanced_agg = build_enhanced_aggregation(relevant, kb) if relevant else {}
    
    val_engine = ValidatorEngine()
    val_methods = kb.get('validation_rules', {}).get('validation_methods', [])
    ev_list = [str(d.get('evidence', [])) for d in extracted]
    val_results = val_engine.run_validations(val_methods, {'evidence': ev_list}, kb['validation_rules'])
    
    comp_j = ComplianceJuror(role=JUROR_ROLE)
    ev_j = EvidenceJuror(role=JUROR_ROLE)
    hse_j = HSEJuror(role=JUROR_ROLE)
    
    comp_r = comp_j.evaluate(kb, enhanced_agg, val_results)
    ev_r = ev_j.evaluate(kb, enhanced_agg, val_results)
    hse_r = hse_j.evaluate(kb, enhanced_agg, val_results)
    
    consensus_agent = ConsensusAgent(role=CONSENSUS_ROLE)
    final = consensus_agent.resolve(comp_r, ev_r, hse_r, kb)
    recommended = snap_to_csms_scale(final.get('recommended_score', 6))
    
    exp_file = os.path.join(folder_path, 'expected_result.json')
    expected = 10
    if os.path.exists(exp_file):
        with open(exp_file, 'r') as f:
            expected = json.load(f).get('expected_score', 10)
    
    elapsed = round(time.time() - t_start, 2)
    
    return {
        'category_id': category_id,
        'expected': expected,
        'predicted': recommended,
        'correct': int(recommended == expected),
        'confidence': final.get('confidence', 0.0),
        'docs': len(extracted),
        'chunks': len(cat_chunks),
        'relevant_chunks': len(relevant),
        'comp_score': snap_to_csms_scale(comp_r.get('score', 6)),
        'ev_score': snap_to_csms_scale(ev_r.get('score', 6)),
        'hse_score': snap_to_csms_scale(hse_r.get('score', 6)),
        'time': elapsed
    }

RUN_BATCH = False

if RUN_BATCH:
    batch_conn = init_evidence_db(os.path.join(OUTPUT_DIR, 'batch_evidence_store.db'))
    batch_results = []
    
    for idx, cat in enumerate(categories, 1):
        print(f'\n[{idx}/{len(categories)}] Category {cat["id"]}: {cat["name"]}')
        try:
            result = score_category_enhanced(cat['id'], cat['path'], batch_conn)
            batch_results.append(result)
            status = 'CORRECT' if result['correct'] else 'WRONG'
            print(f'  Expected={result["expected"]}, Predicted={result["predicted"]} [{status}] in {result["time"]}s')
        except Exception as e:
            print(f'  ERROR: {e}')
            batch_results.append({'category_id': cat['id'], 'expected': 10, 'predicted': 0, 'correct': 0, 'error': str(e)})
    
    df_batch = pd.DataFrame(batch_results)
    accuracy = df_batch['correct'].mean() * 100 if len(df_batch) > 0 else 0
    print(f'\n=== BATCH RESULTS ===')
    print(f'Total categories: {len(batch_results)}')
    print(f'Accuracy: {accuracy:.1f}%')
    display(df_batch)
    
    df_batch.to_csv(os.path.join(OUTPUT_DIR, 'batch_results.csv'), index=False)
    print(f'\nResults saved to {OUTPUT_DIR}/batch_results.csv')
else:
    print('Set RUN_BATCH = True to execute batch evaluation across all categories.')
    print('WARNING: This will make LLM API calls for each category (extraction + embedding + jurors + consensus).')

---
## 16. SQLite Store Exploration

Useful queries for inspecting stored data.

In [ ]:
print('=== SQLite Store Stats ===')

total = conn.execute('SELECT COUNT(*) FROM chunks').fetchone()[0]
print(f'Total chunks: {total}')

by_cat = conn.execute('SELECT category_id, COUNT(*), SUM(char_count) FROM chunks GROUP BY category_id ORDER BY category_id').fetchall()
print(f'\nChunks by category:')
for row in by_cat:
    print(f'  {row[0]:>5s}: {row[1]:>4d} chunks, {row[2]:>8,d} chars')

by_tier = conn.execute('SELECT tier, COUNT(*), AVG(char_count) FROM chunks GROUP BY tier').fetchall()
print(f'\nChunks by tier:')
for row in by_tier:
    print(f'  {row[0]:<30s}: {row[1]:>4d} chunks, avg {row[2]:>6.0f} chars')

with_emb = conn.execute('SELECT COUNT(*) FROM chunks WHERE embedding IS NOT NULL').fetchone()[0]
print(f'\nChunks with embeddings: {with_emb}/{total}')

db_size = os.path.getsize(DB_PATH) / (1024 * 1024)
print(f'Database file size: {db_size:.2f} MB')

print('\n=== Sample FTS5 Keyword Search ===')
fts_results = conn.execute('''
    SELECT c.chunk_id, c.filename, c.tier, SUBSTR(c.text, 1, 200)
    FROM chunks_fts fts
    JOIN chunks c ON c.chunk_id = fts.chunk_id
    WHERE chunks_fts MATCH 'kontraktor OR subkontraktor'
    LIMIT 5
''').fetchall()
for r in fts_results:
    print(f'  [{r[2]}] {r[1]}: {r[3]}...')

---
## 17. Summary & Recommendations

### Pipeline Architecture
| Stage | Component | Enhancement |
|-------|-----------|-------------|
| Extraction | ExtractorRouter + VisionExtractor | Already robust (PDF/OCR/Vision LLM) |
| Aggregation | EvidenceAggregator | **NEW: chunk + embed + SQLite semantic retrieval** |
| Validation | ValidatorEngine (8 validators) | Working as designed |
| Juror Scoring | 3 AI Jurors (Compliance/Evidence/HSE) | Benefits from better aggregation input |
| Consensus | ConsensusAgent (equal-weight + LLM) | Working as designed |
| Recommend | snap_to_csms_scale vs expected | Working as designed |

### Enhancement Benefits
1. **Semantic relevance** — Only top-K evidence chunks (ranked by cosine similarity + keyword FTS5) are fed to jurors
2. **Token efficiency** — Reduced from ~80K chars flat dump to ~7.5K chars (15 chunks × ~500 chars)
3. **Persistent storage** — SQLite stores all chunks + embeddings for future queries
4. **Hybrid retrieval** — Combines semantic similarity with keyword matching via RRF
5. **Traceability** — Every chunk links back to source file, tier, and category

### Model Choice
- Embedding: `text-embedding-3-small` (1536-dim, $0.02/1M tokens)
- Storage: SQLite (zero infrastructure, WAL mode, FTS5 full-text search)
- Aggregation LLM: `gpt-4o-mini` (fast, cheap summarization of retrieved chunks)

### Next Steps
- Integrate `build_enhanced_aggregation()` into `engine/aggregators/evidence_aggregator.py`
- Add `engine/aggregators/chunk_embedder.py` and `engine/aggregators/semantic_retriever.py`
- Consider `text-embedding-3-large` (3072-dim) for higher accuracy if cost is acceptable
- Add batch embedding cache to avoid re-embedding unchanged documents